In [39]:
import arcpy
import os
import csv

In [40]:
# ── GS METHOD: GEODATABASE PATHS ─────────────────────────────────────────────────────────────────────

# Lists of geodatabases containing rasters whose properties will be written to each output CSV - each gdb outputs one CSV with each raster in the gdb as a row in CSV and each raster property as a column; outputs CSVs named by gdb name (i.e., "GEBCO_Gridded2025_15arcsec.gdb" gdb -> GEBCO_Gridded2025_15arcsec.gdb.csv)
gdb_list = [r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Raw_data\US_Caribbean_Bathymetry_raw.gdb"]

# Folder to save output CSVs to (each CSV will be named based on the geodatabase it contains info for)
csv_folder = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\raster_information_csvs\csv_by_geodatabase"

In [41]:
# ── GS METHOD: PRE-BUILD STATISTICS FOR RASTERS BY GEODATABASE ───────────────────────────────────────
# ⭐ skips rasters that already have stats calculated (only applies to rasters without stats)

# loop over each geodatabase
for gdb in gdb_list:
    # point workspace at current gdb so ListRasters looks inside it
    arcpy.env.workspace = gdb
    # loop over every raster in this gdb
    for r in arcpy.ListRasters():
        # build full path to raster
        raster_path = os.path.join(gdb, r)
        # calculate statistics *'SKIP_EXISTING' leaves rasters that already have stats untouched
        try:
            arcpy.management.CalculateStatistics(raster_path, skip_existing="SKIP_EXISTING")
            print(f"✅ Stats OK: {os.path.basename(raster_path)}")
        # print warning if raster fails and continue to next
        except Exception as e:
            print(f"\n⚠️[WARNING] Stats failed for {os.path.basename(raster_path)} - Error: {e} \n")

✅ Stats OK: STTSTJ_2m
✅ Stats OK: STX_2m
✅ Stats OK: PuertoRico_East_2m
✅ Stats OK: PuertoRico_South_2m
✅ Stats OK: PuertoRico_West_2m
✅ Stats OK: PuertoRico_North_2m
✅ Stats OK: US_Caribbean_10m


In [42]:
# ── DICTIONARY OF RASTER PROPERTIES TO INCLUDE IN CSV ────────────────────────────────────────────────

def get_raster_properties(raster_path):
    # get the Describe object (contains name, format, spatial reference, etc.)
    desc = arcpy.Describe(raster_path)
    # get the Raster object, which (contains pixel dimensions and statistics)
    ras = arcpy.Raster(raster_path)
    # build dictionary mapping each property label to its value
    return {
        # store input path so you can trace each raster back to source
        "Path": raster_path,
        # dataset name without full path
        "Name": desc.name,
        # raster format (e.g. FGDBR for file geodatabase raster)
        "Format": desc.format,
        # number of columns (pixels wide)
        "Columns": ras.width,
        # number of rows (pixels tall)
        "Rows": ras.height,
        # number of bands in the raster
        "Bands": ras.bandCount,
        # cell size in the x direction (map units)
        "CellSizeX": ras.meanCellWidth,
        # cell size in the y direction (map units)
        "CellSizeY": ras.meanCellHeight,
        # pixel data type (e.g. short / 16-bit)
        "PixelType": ras.pixelType,
        # NoData value
        "NoData": ras.noDataValue,
        # minimum pixel value from raster statistics (*empty cell in CSV means no stats calculated for raster)
        "Min": ras.minimum,
        # maximum pixel value from raster statistics (*^)
        "Max": ras.maximum,
        # mean pixel value from raster statistics (*^)
        "Mean": ras.mean,
        # standard deviation from raster statistics (*^)
        "StdDev": ras.standardDeviation,
        # name of coordinate system (projected name if projected, else geographic)
        "SpatialRef": desc.spatialReference.name,
        # geographic coordinate system name (underlying datum/GCS)
        "GeogCS": desc.spatialReference.GCS.name if desc.spatialReference.GCS else None,
        # vertical coordinate system name (None if raster has no vertical CS defined)
        "VertCS": desc.spatialReference.VCS.name if desc.spatialReference.VCS else None,
        # spatial extent (xmin, ymin, xmax, ymax) columns
        "Extent_Xmin": ras.extent.XMin,
        "Extent_Ymin": ras.extent.YMin,
        "Extent_Xmax": ras.extent.XMax,
        "Extent_Ymax": ras.extent.YMax,
    }

In [43]:
# ── COMPILE RASTER PROPERTIES INTO CSV DATA ──────────────────────────────────────────────────────────

# Function to write raster properties to CSV (1 raster per row, 1 property per column)
def export_raster_properties(rasters, out_csv):
    # if single path string was passed, wrap it in list so loop works
    if isinstance(rasters, str):
        rasters = [rasters]

    # list that will hold one dictionary (row) per raster
    rows = []
    # loop over every raster path provided
    for r in rasters:
        try:
            # collect properties of raster and add them as a single row
            rows.append(get_raster_properties(r))
        # print warning if raster fails and continue to next raster in gdb
        except Exception as e:
            print(f"⚠️[WARNING] Failure for raster: {r} - Error: {e} \n")

    # if nothing was collected, warn and stop before writing empty file
    if not rows:
        print("⚠️[WARNING] No properties collected (empty CSV file) \n")
        return

    # use keys from the first row as CSV column headers
    fieldnames = list(rows[0].keys())
    # open output CSV for writing (newline="" avoids blank lines on Windows)
    with open(out_csv, "w", newline="") as f:
        # create writer that maps dictionaries to CSV columns
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        # write header row
        writer.writeheader()
        # write one row per raster dictionary
        writer.writerows(rows)

    # print message confirming how many rasters were written and to where
    print(f"✅ Wrote {len(rows)} raster(s) to {out_csv} \n")

In [44]:
# ── GS METHOD: CALL FUNCTION AND EXPORT CSVs (1 CSV PER GEODATABASE) ─────────────────────────────────

# loop over each geodatabase in 'gdb_list' and call 'export_raster_properties'
for gdb in gdb_list:
    # point workspace at current gdb so ListRasters looks inside it
    arcpy.env.workspace = gdb
    # build full path to every raster in this gdb
    raster_list = [os.path.join(gdb, r) for r in arcpy.ListRasters()]
    # name CSV after gdb (e.g. GEBCO_Gridded2025.csv)
    gdb_name = os.path.splitext(os.path.basename(gdb))[0]
    out_csv = os.path.join(csv_folder, f"{gdb_name}.csv")
    # write one combined CSV for all rasters in gdb
    export_raster_properties(raster_list, out_csv)

✅ Wrote 7 raster(s) to C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\raster_information_csvs\csv_by_geodatabase\US_Caribbean_Bathymetry_raw.csv 

